In [1]:
import math
import numpy as np
import pandas as pd
import deepdish as dd
import io
import h5py
from decimal import Decimal, getcontext
getcontext().prec = 40

In [2]:
directory = '/ix/djishnu/Aaron_F/Cleaned_Proteomics_Aaron/20240906/chrombpnet/Results/predictions_contribs'
counts_h5_all = []
profile_h5_all = []
for i in range(0,5):
    c = dd.io.load((directory + '/fold{s}_contribs/fold{s}.counts_scores.h5').format(s = str(i)))
    p = dd.io.load((directory + '/fold{s}_contribs/fold{s}.profile_scores.h5').format(s = str(i)))
    counts_h5_all.append(c)
    profile_h5_all.append(p)

: 

In [2]:
modisco = dd.io.load('modisco/count_modisco_results.h5')

NameError: name 'dd' is not defined

In [3]:
counts_merged = dd.io.load('AM_merged_count_scores.h5')

In [16]:
len(counts_merged['projected_shap']['seq'])

26575

In [20]:
chrbpnet_fold1 = 'chrombpnet_model_fold1/models/chrombpnet_nobias.h5'
test = dd.io.load(chrbpnet_fold1)

In [11]:
def create_combined(arr_folds, savename):
    shap = combine_simple(arr_folds, 'shap')
    pshap = combine_simple(counts_h5_all, 'projected_shap')
    raw = combine_simple(counts_h5_all, 'raw')
    combined = {'projected_shap':{'seq':pshap}, 'shap':{'seq':shap}, 'raw':{'seq':raw}}
    if savename != False:
        dd.io.save(savename, combined)
    return combined

def combine_simple(arr_folds, keyword):
    shaps = []
    for fold in arr_folds:
        shap = fold[keyword]['seq']
        shaps.append(shap)
    sum = shaps[0]
    for i in range(1, len(shaps)):
        sum = sum + shaps[i]
    average = sum/len(shaps)
    return average

def save_dict_to_h5(group, dictionary):
    for key, value in dictionary.items():
        if isinstance(value, dict):
            subgroup = group.create_group(key)
            save_dict_to_h5(subgroup, value)
        else:
            group.create_dataset(key, data=value)

In [25]:
counts_combined = create_combined(counts_h5_all, False)

In [27]:
with h5py.File('AM_merged_count_scores.h5', 'w') as h5file:
    save_dict_to_h5(h5file, counts_combined)

In [29]:
profile_combined = create_combined(profile_h5_all, False)

{'projected_shap': {'seq': array([[[ 0.e+00,  0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00],
          [ 0.e+00,  0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00],
          [ 0.e+00,  0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00],
          [ 0.e+00, -0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00]],
  
         [[ 0.e+00,  0.e+00, -0.e+00, ...,  0.e+00,  0.e+00,  0.e+00],
          [ 0.e+00,  0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00],
          [ 0.e+00,  0.e+00, -6.e-08, ...,  0.e+00,  0.e+00,  0.e+00],
          [ 0.e+00,  0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00]],
  
         [[ 0.e+00,  0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00],
          [ 0.e+00,  0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00],
          [-0.e+00,  0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00],
          [ 0.e+00, -0.e+00, -0.e+00, ...,  0.e+00,  0.e+00,  0.e+00]],
  
         ...,
  
         [[ 0.e+00,  0.e+00,  0.e+00, ...,  0.e+00,  0.e+00,  0.e+00],
          [ 0.e+00,  0.

In [30]:
with h5py.File('AM_merged_profile_scores.h5', 'w') as h5file:
    save_dict_to_h5(h5file, profile_combined)